<a href="https://colab.research.google.com/github/neohack22/ebw3nt/blob/main/apprentissage/Correction_text_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install keras keras-hub --upgrade -q

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "jax"

In [ ]:
# @title
import os
from IPython.core.magic import register_cell_magic

@register_cell_magic
def backend(line, cell):
    current, required = os.environ.get("KERAS_BACKEND", ""), line.split()[-1]
    if current == required:
        get_ipython().run_cell(cell)
    else:
        print(
            f"This cell requires the {required} backend. To run it, change KERAS_BACKEND to "
            f"\"{required}\" at the top of the notebook, restart the runtime, and rerun the notebook."
        )

## Génération de texte

### Bref historique de la génération de séquences

### Entraînement d'un mini-GPT

In [ ]:
import os

# Free up more GPU memory on the Jax and TensorFlow backends.
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "1.00"

Réimporte `os` puis fixe `XLA_PYTHON_CLIENT_MEM_FRACTION` à `1.00` pour mieux gérer la mémoire GPU avec JAX/TensorFlow. L’idée est de limiter certains problèmes de réservation mémoire pendant l’entraînement de modèles assez lourds.

### Charger et inspecter un corpus textuel

Téléchargez le mini-corpus utilisé pour l’entraînement du modèle, extrayez-le localement, puis affichez le début d’un fichier texte pour vérifier son contenu.

In [ ]:
import keras
import pathlib

extract_dir = keras.utils.get_file(
    fname="mini-c4",
    origin=(
        "https://hf.co/datasets/mattdangerw/mini-c4/resolve/main/mini-c4.zip"
    ),
    extract=True,
)
extract_dir = pathlib.Path(extract_dir) / "mini-c4"

In [ ]:
with open(extract_dir / "shard0.txt", "r") as f:
    print(f.readline().replace("\\n", "\n")[:100])

Télécharge et extrait localement un mini-corpus texte (`mini-c4`), puis ouvre un des fichiers pour afficher le début du contenu. Ce bloc sert à vérifier que les données d’entraînement sont bien présentes et lisibles avant de construire le pipeline.

### Tokeniser du texte avec SentencePiece

Téléchargez le vocabulaire du tokenizer, instanciez un tokenizer SentencePiece, puis testez :
- la conversion d’une phrase en tokens ;
- la reconstruction du texte à partir d’une séquence d’identifiants.

In [ ]:
import keras_hub
import numpy as np

vocabulary_file = keras.utils.get_file(
    origin="https://hf.co/mattdangerw/spiece/resolve/main/vocabulary.proto",
)
tokenizer = keras_hub.tokenizers.SentencePieceTokenizer(vocabulary_file)

In [ ]:
tokenizer.tokenize("The quick brown fox.")

In [ ]:
tokenizer.detokenize([450, 4996, 17354, 1701, 29916, 29889])

Télécharge le fichier de vocabulaire du tokenizer, crée un tokenizer `SentencePiece`, puis teste deux opérations de base : transformer du texte en tokens et refaire du texte à partir d’identifiants. C’est une étape clé, car le modèle ne lit pas du texte brut : il travaille sur des ids numériques.

### Préparer les données pour un modèle de langage causal

Construisez un pipeline de lecture des fichiers texte qui :
- remplace les caractères d’échappement par de vrais retours à la ligne ;
- tokenise chaque ligne ;
- ajoute un token de fin de texte ;
- regroupe les données en séquences de longueur fixe ;
- construit des couples `(entrée, cible)` décalés d’un token ;
- prépare des batches prêts pour l’entraînement.

In [ ]:
import tensorflow as tf

batch_size = 64
sequence_length = 256
suffix = np.array([tokenizer.token_to_id("<|endoftext|>")])

def read_file(filename):
    ds = tf.data.TextLineDataset(filename)
    ds = ds.map(lambda x: tf.strings.regex_replace(x, r"\\n", "\n"))
    ds = ds.map(tokenizer, num_parallel_calls=8)
    return ds.map(lambda x: tf.concat([x, suffix], -1))

files = [str(file) for file in extract_dir.glob("*.txt")]
ds = tf.data.Dataset.from_tensor_slices(files)
ds = ds.interleave(read_file, cycle_length=32, num_parallel_calls=32)
ds = ds.rebatch(sequence_length + 1, drop_remainder=True)
ds = ds.map(lambda x: (x[:-1], x[1:]))
ds = ds.batch(batch_size).prefetch(8)

Construit le pipeline de préparation des données pour un modèle auto-régressif.

- `batch_size` fixe le nombre de séquences traitées ensemble.
- `sequence_length` fixe la longueur des séquences d’entrée.
- `suffix` contient le token de fin de texte.

La fonction `read_file()` :
- lit un fichier ligne par ligne ;
- remplace les `\\n` littéraux par de vrais retours à la ligne ;
- tokenise chaque ligne ;
- ajoute un token de fin.

Ensuite :
- on récupère tous les fichiers texte ;
- on les mélange dans un dataset unique avec `interleave` ;
- on regroupe les tokens en blocs de taille fixe `sequence_length + 1` ;
- on fabrique des couples `(entrée, cible)` décalés d’un token : l’entrée contient tous les tokens sauf le dernier, la cible tous les tokens sauf le premier ;
- on forme des batches et on précharge les données avec `prefetch`.

C’est exactement ce qu’il faut pour entraîner un modèle à prédire le prochain token.

### Créer un jeu d’entraînement et un jeu de validation

À partir du pipeline précédent, définissez :
- le nombre total de batches ;
- un sous-ensemble réservé à la validation ;
- un sous-ensemble réservé à l’entraînement.

Faites en sorte que ces ensembles puissent être réutilisés en boucle pendant l’apprentissage.

In [ ]:
num_batches = 58746
num_val_batches = 500
num_train_batches = num_batches - num_val_batches
val_ds = ds.take(num_val_batches).repeat()
train_ds = ds.skip(num_val_batches).repeat()

Sépare le dataset en deux parties :
- `val_ds` pour la validation ;
- `train_ds` pour l’entraînement.

`take()` prend les premiers batches pour la validation, `skip()` garde le reste pour l’entraînement.  
`repeat()` permet de boucler indéfiniment sur ces datasets pendant l’apprentissage, ce qui évite de reconstruire le pipeline à chaque époque.

#### Construction du modèle

### Implémenter un bloc décodeur Transformer

Créez une couche personnalisée représentant un bloc décodeur Transformer avec :
- une auto-attention multi-tête causale ;
- une connexion résiduelle ;
- une normalisation ;
- un réseau feed-forward ;
- du dropout.

In [ ]:
from keras import layers

class TransformerDecoder(keras.Layer):
    def __init__(self, hidden_dim, intermediate_dim, num_heads):
        super().__init__()
        key_dim = hidden_dim // num_heads
        self.self_attention = layers.MultiHeadAttention(
            num_heads, key_dim, dropout=0.1
        )
        self.self_attention_layernorm = layers.LayerNormalization()
        self.feed_forward_1 = layers.Dense(intermediate_dim, activation="relu")
        self.feed_forward_2 = layers.Dense(hidden_dim)
        self.feed_forward_layernorm = layers.LayerNormalization()
        self.dropout = layers.Dropout(0.1)

    def call(self, inputs):
        residual = x = inputs
        x = self.self_attention(query=x, key=x, value=x, use_causal_mask=True)
        x = self.dropout(x)
        x = x + residual
        x = self.self_attention_layernorm(x)
        residual = x
        x = self.feed_forward_1(x)
        x = self.feed_forward_2(x)
        x = self.dropout(x)
        x = x + residual
        x = self.feed_forward_layernorm(x)
        return x

Définit un bloc décodeur Transformer personnalisé.

À l’intérieur :
- `MultiHeadAttention` réalise l’auto-attention ;
- `use_causal_mask=True` empêche chaque position de voir les tokens futurs ;
- il y a ensuite une connexion résiduelle puis une normalisation ;
- puis un petit réseau feed-forward (`Dense -> Dense`) ;
- puis une nouvelle connexion résiduelle, une normalisation et du dropout.

Dans `call()` :
1. le bloc applique l’attention causale ;
2. ajoute le résultat à l’entrée initiale ;
3. normalise ;
4. applique le feed-forward ;
5. rajoute encore la sortie au résiduel ;
6. renormalise.

C’est la brique de base d’un GPT : attention causale + MLP + résidus + normalisation.

### Ajouter des embeddings de tokens et de positions

Créez une couche personnalisée qui :
- associe un vecteur à chaque token ;
- associe un vecteur à chaque position dans la séquence ;
- additionne les deux pour produire la représentation d’entrée du modèle.

Ajoutez aussi un mode permettant de projeter les représentations cachées vers l’espace du vocabulaire.

In [ ]:
from keras import ops

class PositionalEmbedding(keras.Layer):
    def __init__(self, sequence_length, input_dim, output_dim):
        super().__init__()
        self.token_embeddings = layers.Embedding(input_dim, output_dim)
        self.position_embeddings = layers.Embedding(sequence_length, output_dim)

    def call(self, inputs, reverse=False):
        if reverse:
            token_embeddings = self.token_embeddings.embeddings
            return ops.matmul(inputs, ops.transpose(token_embeddings))
        positions = ops.cumsum(ops.ones_like(inputs), axis=-1) - 1
        embedded_tokens = self.token_embeddings(inputs)
        embedded_positions = self.position_embeddings(positions)
        return embedded_tokens + embedded_positions

Définit une couche d’embedding positionnel.

Elle combine deux informations :
- l’identité du token (`token_embeddings`) ;
- sa position dans la séquence (`position_embeddings`).

Dans le mode normal :
- on calcule les positions ;
- on récupère l’embedding de chaque token ;
- on récupère l’embedding de chaque position ;
- on additionne les deux.

Dans le mode `reverse=True` :
- on réutilise la matrice d’embedding des tokens pour projeter les représentations cachées vers l’espace du vocabulaire ;
- autrement dit, on transforme des vecteurs cachés en scores sur tous les mots possibles.

C’est une technique classique de partage de poids entre embeddings d’entrée et projection de sortie.

### Construire un mini modèle GPT

Assemblez un modèle de langage auto-régressif en définissant :
- la taille du vocabulaire ;
- la dimension cachée ;
- la dimension intermédiaire ;
- le nombre de têtes d’attention ;
- le nombre de couches Transformer.

Le modèle doit prendre une séquence d’identifiants en entrée et produire, pour chaque position, une distribution sur le vocabulaire.

In [ ]:
keras.config.set_dtype_policy("mixed_float16")

vocab_size = tokenizer.vocabulary_size()
hidden_dim = 512
intermediate_dim = 2056
num_heads = 8
num_layers = 8

inputs = keras.Input(shape=(None,), dtype="int32", name="inputs")
embedding = PositionalEmbedding(sequence_length, vocab_size, hidden_dim)
x = embedding(inputs)
x = layers.LayerNormalization()(x)
for i in range(num_layers):
    x = TransformerDecoder(hidden_dim, intermediate_dim, num_heads)(x)
outputs = embedding(x, reverse=True)
mini_gpt = keras.Model(inputs, outputs)

Assemble le mini-GPT.

- `mixed_float16` active des calculs plus légers en mémoire.
- Les hyperparamètres définissent la taille du modèle : vocabulaire, dimension cachée, nombre de têtes, nombre de couches, etc.
- `inputs` représente une séquence d’ids.
- `embedding(inputs)` transforme ces ids en vecteurs enrichis par la position.
- une normalisation est appliquée ;
- puis plusieurs blocs `TransformerDecoder` sont empilés ;
- enfin `embedding(x, reverse=True)` projette les représentations finales vers des logits sur le vocabulaire.

Le modèle obtenu prédit, pour chaque position, quel token a le plus de chances de venir ensuite.

#### Préentraînement du modèle

### Définir un scheduler avec phase de warmup

Créez une classe représentant une stratégie de taux d’apprentissage qui augmente progressivement au début de l’entraînement jusqu’à atteindre une valeur cible.

Tracez ensuite l’évolution du taux d’apprentissage en fonction du nombre d’itérations.

In [ ]:
class WarmupSchedule(keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self):
        self.rate = 2e-4
        self.warmup_steps = 1_000.0

    def __call__(self, step):
        step = ops.cast(step, dtype="float32")
        scale = ops.minimum(step / self.warmup_steps, 1.0)
        return self.rate * scale

In [ ]:
import matplotlib.pyplot as plt

schedule = WarmupSchedule()
x = range(0, 5_000, 100)
y = [ops.convert_to_numpy(schedule(step)) for step in x]
plt.plot(x, y)
plt.xlabel("Train Step")
plt.ylabel("Learning Rate")
plt.show()

Crée un scheduler de taux d’apprentissage avec *warmup*.

- `rate` est la valeur cible du learning rate ;
- `warmup_steps` est le nombre d’étapes pendant lesquelles on monte progressivement vers cette valeur.

Dans `__call__` :
- on convertit `step` en flottant ;
- on calcule une échelle entre 0 et 1 ;
- le learning rate augmente linéairement jusqu’à atteindre `2e-4`.

La partie `matplotlib` trace ensuite cette montée progressive.  
Le but du warmup est de stabiliser le début de l’entraînement, surtout pour les Transformers.

In [ ]:
# ⚠️REMARQUE⚠️ : Si vous pouvez exécuter le code suivant avec un GPU Colab Pro, nous vous le recommandons.

# L'appel à fit() peut prendre plusieurs heures avec les GPU gratuits. Vous pouvez également
# réduire steps_per_epoch pour tester le code avec un modèle moins entraîné.

### Préentraîner le modèle de langage

Compilez le modèle avec :
- un optimiseur Adam utilisant le scheduler précédent ;
- une fonction de perte adaptée à une classification multi-classes sur les tokens ;
- une métrique de précision.

Lancez ensuite l’entraînement sur les données préparées en précisant le nombre d’époques, le nombre d’itérations par époque et les étapes de validation.

In [ ]:
num_epochs = 8
steps_per_epoch = num_train_batches // num_epochs
validation_steps = num_val_batches

mini_gpt.compile(
    optimizer=keras.optimizers.Adam(schedule),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)
mini_gpt.fit(
    train_ds,
    validation_data=val_ds,
    epochs=num_epochs,
    steps_per_epoch=steps_per_epoch,
    validation_steps=validation_steps,
)

Fixe les paramètres de l’entraînement :
- `num_epochs` : nombre d’époques ;
- `steps_per_epoch` : nombre de batches vus par époque ;
- `validation_steps` : nombre de batches de validation.

Puis :
- compile le modèle avec Adam ;
- utilise une entropie croisée adaptée à des cibles entières (`SparseCategoricalCrossentropy`) ;
- indique que la sortie est sous forme de logits ;
- suit aussi l’accuracy.

Enfin, `fit()` lance le préentraînement sur `train_ds` avec évaluation régulière sur `val_ds`.

#### Décodage génératif

### Générer du texte à partir d’un prompt

Écrivez une fonction qui :
- tokenise un prompt ;
- prédit itérativement le token suivant ;
- ajoute ce token à la séquence ;
- reconstruit ensuite le texte généré.

In [ ]:
def generate(prompt, max_length=64):
    tokens = list(ops.convert_to_numpy(tokenizer(prompt)))
    prompt_length = len(tokens)
    for _ in range(max_length - prompt_length):
        prediction = mini_gpt(ops.convert_to_numpy([tokens]))
        prediction = ops.convert_to_numpy(prediction[0, -1])
        tokens.append(np.argmax(prediction).item())
    return tokenizer.detokenize(tokens)

Implémente une génération auto-régressive simple.

1. le prompt est tokenisé ;
2. on mémorise sa longueur ;
3. tant qu’on n’a pas atteint `max_length`, on envoie la séquence actuelle au modèle ;
4. on récupère la prédiction du dernier token ;
5. on choisit le token le plus probable avec `argmax` ;
6. on l’ajoute à la séquence ;
7. à la fin, on reconvertit la séquence en texte.

C’est la version la plus directe de la génération : à chaque étape, le modèle prédit le prochain token à partir de tout ce qu’il a déjà produit.

In [ ]:
prompt = "A piece of advice"
generate(prompt)

### Optimiser la génération de texte

Proposez une seconde version de la génération qui préalloue une séquence de longueur fixe, remplit progressivement les positions générées, puis comparez son temps d’exécution à l’aide d’un petit benchmark.

In [ ]:
def compiled_generate(prompt, max_length=64):
    tokens = list(ops.convert_to_numpy(tokenizer(prompt)))
    prompt_length = len(tokens)
    tokens = tokens + [0] * (max_length - prompt_length)
    for i in range(prompt_length, max_length):
        prediction = mini_gpt.predict(np.array([tokens]), verbose=0)
        prediction = prediction[0, i - 1]
        tokens[i] = np.argmax(prediction).item()
    return tokenizer.detokenize(tokens)

In [ ]:
import timeit
tries = 10
timeit.timeit(lambda: compiled_generate(prompt), number=tries) / tries

Propose une version plus efficace de la génération.

Cette fois :
- la séquence complète est préallouée à longueur fixe ;
- seules les positions générées sont remplies progressivement ;
- on évite donc certaines reconstructions répétées de listes Python.

`mini_gpt.predict(..., verbose=0)` est utilisé dans la boucle pour produire les logits.  
Le `timeit` mesure ensuite le temps moyen d’exécution sur plusieurs essais pour comparer les performances.

#### Stratégies d'échantillonnage

### Tester différentes stratégies de décodage

Modifiez la génération de texte pour permettre le choix d’une stratégie d’échantillonnage.

Implémentez et comparez au moins :
- une stratégie gloutonne (*greedy search*) ;
- un échantillonnage aléatoire avec température ;
- une stratégie *top-k*.

In [ ]:
def compiled_generate(prompt, sample_fn, max_length=64):
    tokens = list(ops.convert_to_numpy(tokenizer(prompt)))
    prompt_length = len(tokens)
    tokens = tokens + [0] * (max_length - prompt_length)
    for i in range(prompt_length, max_length):
        prediction = mini_gpt.predict(np.array([tokens]), verbose=0)
        prediction = prediction[0, i - 1]
        next_token = ops.convert_to_numpy(sample_fn(prediction))
        tokens[i] = np.array(next_token).item()
    return tokenizer.detokenize(tokens)

Redéfinit `compiled_generate()` pour rendre la stratégie de choix du prochain token paramétrable via `sample_fn`.

La logique reste la même, mais au lieu de toujours prendre `argmax`, on délègue la sélection du token suivant à une fonction externe.  
Ça permet de tester différentes stratégies de décodage sans réécrire toute la génération.

In [ ]:
def greedy_search(preds):
    return ops.argmax(preds)

compiled_generate(prompt, greedy_search)

`greedy_search()` renvoie simplement l’indice du token le plus probable.  
C’est la stratégie gloutonne : simple et déterministe, mais elle peut produire des textes répétitifs ou peu variés.

Appelle la génération avec la stratégie gloutonne pour observer le comportement du modèle quand on prend systématiquement le meilleur score local.

In [ ]:
def random_sample(preds, temperature=1.0):
    preds = preds / temperature
    return keras.random.categorical(preds[None, :], num_samples=1)[0]

`random_sample()` ajoute de l’aléatoire dans la génération.

- on divise les logits par `temperature` ;
- puis on tire un token aléatoirement selon la distribution obtenue.

Effet de `temperature` :
- grande température → plus de diversité ;
- petite température → plus conservateur, plus proche du greedy.

In [ ]:
compiled_generate(prompt, random_sample)

Teste l’échantillonnage aléatoire avec la température par défaut (`1.0`).  
Le résultat est généralement plus varié qu’en greedy.

In [ ]:
from functools import partial
compiled_generate(prompt, partial(random_sample, temperature=2.0))

Teste une température élevée (`2.0`).  
La distribution est aplatie, donc le modèle ose plus de tokens peu probables : le texte devient souvent plus créatif, mais aussi plus instable.

In [ ]:
compiled_generate(prompt, partial(random_sample, temperature=0.8))

Teste une température modérée (`0.8`).  
On garde un peu d’aléatoire sans trop dégrader la cohérence.

In [ ]:
compiled_generate(prompt, partial(random_sample, temperature=0.2))

Teste une température basse (`0.2`).  
La génération devient très concentrée sur les tokens les plus probables, donc plus prévisible et proche du greedy.

In [ ]:
def top_k(preds, k=5, temperature=1.0):
    preds = preds / temperature
    top_preds, top_indices = ops.top_k(preds, k=k, sorted=False)
    choice = keras.random.categorical(top_preds[None, :], num_samples=1)[0]
    return ops.take_along_axis(top_indices, choice, axis=-1)

`top_k()` limite le choix aux `k` meilleurs tokens.

- on applique éventuellement une température ;
- on garde seulement les `k` plus grands scores ;
- on échantillonne à l’intérieur de ce sous-ensemble ;
- puis on remappe vers l’id original du token choisi.

L’idée est de conserver de la diversité tout en évitant les choix trop improbables.

In [ ]:
compiled_generate(prompt, partial(top_k, k=5))

Teste le `top-k` avec `k=5`.  
Le choix est restreint aux 5 meilleures options, ce qui garde généralement une bonne cohérence.

In [ ]:
compiled_generate(prompt, partial(top_k, k=20))

Teste le `top-k` avec `k=20`.  
Le modèle a plus de liberté qu’avec `k=5`, donc les sorties sont en général plus variées.

In [ ]:
compiled_generate(prompt, partial(top_k, k=5, temperature=0.5))

Teste une combinaison `top-k` + température basse (`k=5`, `temperature=0.5`).  
On reste dans les meilleurs candidats, mais avec une distribution encore plus concentrée.

### Utiliser un modèle de langage préentraîné

Connectez-vous au service requis, chargez un modèle causal préentraîné, affichez son résumé, puis utilisez-le pour générer du texte à partir de plusieurs prompts.

In [ ]:
import kagglehub

kagglehub.login()

In [ ]:
gemma_lm = keras_hub.models.CausalLM.from_preset(
    "gemma3_1b",
    dtype="float32",
)

In [ ]:
gemma_lm.summary(line_length=80)

In [ ]:
gemma_lm.compile(sampler="greedy")
gemma_lm.generate("A piece of advice", max_length=40)

In [ ]:
gemma_lm.generate("How can I make brownies?", max_length=40)

In [ ]:
gemma_lm.generate(
    "The following brownie recipe is easy to make in just a few "
    "steps.\n\nYou can start by",
    max_length=40,
)

In [ ]:
gemma_lm.generate(
    "Tell me about the 542nd president of the United States.",
    max_length=40,
)

Connecte l’environnement à Kaggle, puis charge un modèle causal préentraîné `gemma3_1b`.

- `from_preset(...)` récupère l’architecture et les poids déjà appris ;
- `dtype="float32"` choisit le type numérique ;
- `summary()` affiche la structure du modèle ;
- `compile(sampler="greedy")` configure la stratégie de génération ;
- `generate(...)` teste le modèle sur plusieurs prompts.

Ici, on ne part plus d’un mini-modèle entraîné maison : on exploite directement un vrai LLM préentraîné.

#### Instruction fine-tuning

### Préparer des données pour un fine-tuning instructionnel

Téléchargez un jeu de données d’instructions/réponses, filtrez les exemples selon un critère donné, puis construisez :
- un format de prompt d’instruction ;
- un format de réponse cible ;
- un dataset TensorFlow prêt à être batché.

In [ ]:
import json

PROMPT_TEMPLATE = """"[instruction]\n{}[end]\n[response]\n"""
RESPONSE_TEMPLATE = """{}[end]"""

dataset_path = keras.utils.get_file(
    origin=(
        "https://hf.co/datasets/databricks/databricks-dolly-15k/"
        "resolve/main/databricks-dolly-15k.jsonl"
    ),
)
data = {"prompts": [], "responses": []}
with open(dataset_path) as file:
    for line in file:
        features = json.loads(line)
        if features["context"]:
            continue
        data["prompts"].append(PROMPT_TEMPLATE.format(features["instruction"]))
        data["responses"].append(RESPONSE_TEMPLATE.format(features["response"]))

Définit les formats de prompt/réponse pour le fine-tuning instructionnel, puis charge le dataset Dolly.

- `PROMPT_TEMPLATE` entoure l’instruction avec des balises ;
- `RESPONSE_TEMPLATE` fait pareil pour la réponse ;
- le fichier `.jsonl` est téléchargé ;
- chaque ligne est lue comme un exemple indépendant ;
- les exemples avec `context` sont ignorés ;
- on construit deux listes : prompts et réponses.

Enfin :
- on crée un `tf.data.Dataset` ;
- on mélange les exemples ;
- on batch par 2 ;
- on sépare validation et entraînement.

Le but est de reformater les données pour apprendre au modèle à répondre à des consignes.

In [ ]:
data["prompts"][0]

In [ ]:
data["responses"][0]

In [ ]:
ds = tf.data.Dataset.from_tensor_slices(data).shuffle(2000).batch(2)
val_ds = ds.take(100)
train_ds = ds.skip(100)

### Vérifier la sortie du préprocesseur

Utilisez le préprocesseur du modèle pour transformer un batch de données textuelles, puis examinez la forme des tenseurs produits :
- identifiants de tokens ;
- masque de padding ;
- cibles ;
- poids d’échantillons.

In [ ]:
preprocessor = gemma_lm.preprocessor
preprocessor.sequence_length = 512
batch = next(iter(train_ds))
x, y, sample_weight = preprocessor(batch)
x["token_ids"].shape

Récupère le préprocesseur du modèle Gemma et fixe une longueur maximale de séquence.

Puis, sur un batch :
- `x` contient les entrées prétraitées ;
- `y` contient les cibles ;
- `sample_weight` contient les poids appliqués à la perte.

Les affichages de `shape` servent à vérifier que tout est bien aligné :
- `token_ids` : ids des tokens ;
- `padding_mask` : masque indiquant quelles positions sont réelles ou du padding ;
- `y` : cibles à prédire ;
- `sample_weight` : pondération des positions utiles.

L'avant-dernière ligne compare le début des ids d’entrée et des cibles pour montrer le décalage classique du next-token prediction.

In [ ]:
x["padding_mask"].shape

In [ ]:
y.shape

In [ ]:
sample_weight.shape

In [ ]:
x["token_ids"][0, :5], y[0, :5]

#### Low-Rank Adaptation (LoRA)

### Adapter le modèle avec LoRA

Activez une adaptation de type **LoRA** sur le backbone du modèle, affichez à nouveau le résumé du modèle, puis compilez et entraînez-le sur le dataset d’instruction fine-tuning.

In [ ]:
gemma_lm.backbone.enable_lora(rank=8)

In [ ]:
gemma_lm.summary(line_length=80)

In [ ]:
gemma_lm.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.Adam(5e-5),
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy()],
)
gemma_lm.fit(train_ds, validation_data=val_ds, epochs=1)

Active LoRA sur le backbone du modèle avec un rang 8.

LoRA permet de fine-tuner un grand modèle en n’ajoutant qu’un petit nombre de paramètres adaptatifs, au lieu de modifier tous les poids.  
Ensuite :
- `summary()` permet de voir la nouvelle structure ;
- `compile(...)` définit perte, optimiseur et métrique ;
- `fit(...)` lance l’entraînement instructionnel.

L’intérêt principal est de réduire fortement le coût mémoire et le nombre de paramètres à mettre à jour.

### Évaluer qualitativement le modèle après adaptation

Générez plusieurs réponses à partir de consignes rédigées dans le format attendu par le modèle, afin d’observer l’effet du fine-tuning sur ses réponses.

In [ ]:
gemma_lm.generate(
    "[instruction]\nHow can I make brownies?[end]\n"
    "[response]\n",
    max_length=512,
)

In [ ]:
gemma_lm.generate(
    "[instruction]\nWhat is a proper noun?[end]\n"
    "[response]\n",
    max_length=512,
)

In [ ]:
gemma_lm.generate(
    "[instruction]\nWho is the 542nd president of the United States?[end]\n"
    "[response]\n",
    max_length=512,
)

Teste qualitativement le modèle après adaptation LoRA.

Chaque appel `generate(...)` envoie une consigne déjà formatée selon le schéma attendu :
- balise `[instruction]` ;
- texte de la consigne ;
- balise `[response]`.

Le but est de voir si le modèle répond mieux au style instructionnel après fine-tuning.

### Poursuivre les LLMs


#### Reinforcement Learning with Human Feedback (RLHF)

##### Utilisation d'un chatbot entraîné avec RLHF

In [ ]:
# ⚠️REMARQUE⚠️ : Si vous utilisez les GPU Colab gratuits, vous devrez

# redémarrer votre environnement d'exécution et exécuter le notebook à partir d'ici pour libérer de la mémoire pour
# ce modèle à 4 milliards de paramètres.
import os

os.environ["KERAS_BACKEND"] = "jax"
# Free up more GPU memory on the Jax and TensorFlow backends.
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "1.00"

import keras
import keras_hub
import kagglehub
import numpy as np

kagglehub.login()

### Utiliser une version instructionnelle plus avancée du modèle

Chargez une version conversationnelle du modèle, définissez un format de prompt de dialogue, puis testez-le sur plusieurs questions afin d’observer son comportement en mode chatbot.

In [ ]:
gemma_lm = keras_hub.models.CausalLM.from_preset(
    "gemma3_instruct_4b",
    dtype="bfloat16",
)

In [ ]:
PROMPT_TEMPLATE = """<start_of_turn>user
{}<end_of_turn>
<start_of_turn>model
"""

In [ ]:
prompt = "Why can't you assign values in Jax tensors? Be brief!"
gemma_lm.generate(PROMPT_TEMPLATE.format(prompt), max_length=512)

In [ ]:
prompt = "Who is the 542nd president of the United States?"
gemma_lm.generate(PROMPT_TEMPLATE.format(prompt), max_length=512)

Charge `gemma3_instruct_4b`, une version instructionnelle plus avancée, pensée pour le dialogue.

Le `PROMPT_TEMPLATE` simule une conversation structurée avec tours `user` et `model`.  
Ensuite, deux prompts sont testés :
- une question technique sur JAX ;
- une question volontairement absurde sur le “542e président”.

Le but est d’observer le comportement du modèle dans un cadre chatbot.

#### LLMs multimodaux

### Interroger un modèle multimodal à partir d’une image

Téléchargez une image, affichez-la, puis utilisez un modèle multimodal pour répondre à des questions portant sur le contenu visuel de cette image.

In [ ]:
import matplotlib.pyplot as plt

image_url = (
    "https://github.com/mattdangerw/keras-nlp-scripts/"
    "blob/main/learned-python.png?raw=true"
)
image_path = keras.utils.get_file(origin=image_url)

image = np.array(keras.utils.load_img(image_path))
plt.axis("off")
plt.imshow(image)
plt.show()

In [ ]:
gemma_lm.preprocessor.max_images_per_prompt = 1
gemma_lm.preprocessor.sequence_length = 512
prompt = "What is going on in this image? Be concise!<start_of_image>"
gemma_lm.generate({
    "prompts": PROMPT_TEMPLATE.format(prompt),
    "images": [image],
})

In [ ]:
prompt = "What is the snake wearing?<start_of_image>"
gemma_lm.generate({
    "prompts": PROMPT_TEMPLATE.format(prompt),
    "images": [image],
})

Télécharge une image, l’affiche, puis prépare le modèle multimodal à accepter une image par prompt.

Ensuite :
- `max_images_per_prompt = 1` fixe la capacité image ;
- `sequence_length = 512` fixe la longueur maximale ;
- les prompts incluent la balise `<start_of_image>` ;
- `generate({...})` reçoit à la fois le texte du prompt et l’image.

Certains LLM modernes peuvent traiter texte + image dans la même requête.

##### Modèles de fondation

#### Retrieval Augmented Generation (RAG)

#### Modèles de « raisonnement »

### Tester le modèle sur une question de raisonnement simple

Soumettez au modèle un problème court demandant un raisonnement arithmétique, puis générez plusieurs réponses afin d’observer la variabilité des sorties lorsque l’échantillonnage est aléatoire.

In [ ]:
prompt = """Judy wrote a 2-page letter to 3 friends twice a week for 3 months.
How many letters did she write?
Be brief, and add "ANSWER:" before your final answer."""

gemma_lm.compile(sampler="random")

In [ ]:
gemma_lm.generate(PROMPT_TEMPLATE.format(prompt))

In [ ]:
gemma_lm.generate(PROMPT_TEMPLATE.format(prompt))

Prépare un petit problème de raisonnement arithmétique, puis configure le modèle avec un échantillonnage aléatoire (`sampler="random"`).

Les deux générations successives permettent de comparer plusieurs sorties possibles pour le même prompt et de voir la variabilité induite par l’aléatoire.

### Rechercher ce qu’est la génération augmentée par récupération (RAG)

Expliquez en quelques lignes le principe du **Retrieval Augmented Generation (RAG)** et indiquez en quoi cette approche peut compléter un modèle de langage préentraîné.

- le **RAG** : récupérer de l’information externe pour aider le modèle à répondre avec des connaissances plus fraîches ou plus précises ;

### Expliquer la notion de modèle de fondation

Définissez ce qu’on appelle un **foundation model** et donnez un exemple d’usage possible dans le cadre du traitement du langage.

- le **foundation model** : grand modèle généraliste préentraîné sur d’énormes volumes de données, qu’on adapte ensuite à des tâches précises ;

### Quelle sera la prochaine étape pour les LLM ?

- les **prochaines étapes des LLM** : modèles multimodaux, meilleurs raisonnements, adaptation plus efficace, intégration d’outils, RAG, RLHF, etc.